# Fine-tune Qwen2.5-1.5B for OLX Car Description Extraction

Training data: 1047 Portuguese car listing descriptions annotated by Claude Opus 4.6.

**Steps:**
1. Upload `training_data.jsonl`
2. Fine-tune with Unsloth + QLoRA
3. Export to GGUF for Ollama
4. Download and deploy

In [ ]:
# 1. Install Unsloth (optimized for free Colab T4)
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
# 2. Upload training data
from google.colab import files
uploaded = files.upload()  # upload training_data.jsonl

In [ ]:
# 3. Load and prepare dataset
import json
from datasets import Dataset

data = []
with open("training_data.jsonl") as f:
    for line in f:
        data.append(json.loads(line))

print(f"Loaded {len(data)} training examples")
print(f"Sample input (first 200 chars): {data[0]['messages'][0]['content'][:200]}")
print(f"Sample output (first 200 chars): {data[0]['messages'][1]['content'][:200]}")

In [ ]:
# 4. Load model with Unsloth (4-bit quantized)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length=2048,
    dtype=None,  # auto-detect
    load_in_4bit=True,
)

print(f"Model loaded. Parameters: {model.num_parameters():,}")

In [ ]:
# 5. Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                     # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,           # optimized, no dropout
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less VRAM
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

In [ ]:
# 6. Format dataset for chat template
def format_chat(example):
    """Apply Qwen2.5 chat template to training example."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = Dataset.from_list(data)
dataset = dataset.map(format_chat)

# Check token lengths
lengths = [len(tokenizer.encode(x["text"])) for x in dataset]
print(f"Token lengths — min: {min(lengths)}, max: {max(lengths)}, avg: {sum(lengths)/len(lengths):.0f}")
print(f"\nFormatted sample (first 300 chars):\n{dataset[0]['text'][:300]}")

In [ ]:
# 7. Train
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,  # effective batch = 16
        warmup_steps=20,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="outputs",
        save_strategy="epoch",
    ),
)

print(f"Training {len(dataset)} examples, 3 epochs...")
stats = trainer.train()
print(f"\nDone! Final loss: {stats.training_loss:.4f}")

In [ ]:
# 8. Quick test before export
FastLanguageModel.for_inference(model)

test_desc = """Vendo BMW 320d de 2015, 180cv, 150.000km.
Carro nacional, unico dono, revisoes sempre na marca.
Sem acidentes. Pneus novos, travoes recentes.
Vendo por motivo de compra de carro novo."""

messages = [{"role": "user", "content": test_desc}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(**inputs, max_new_tokens=512, temperature=0.1)
response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("Test output:")
print(response)

In [ ]:
# 9. Export to GGUF for Ollama
# Q4_K_M — best balance of speed and quality for M1 8GB
model.save_pretrained_gguf(
    "car-parser-1.5b",
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF exported!")

In [ ]:
# 10. Create Ollama Modelfile
modelfile = """FROM ./car-parser-1.5b-Q4_K_M.gguf

PARAMETER temperature 0.1
PARAMETER num_predict 512
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|endoftext|>"
"""

with open("car-parser-1.5b/Modelfile", "w") as f:
    f.write(modelfile)

print("Modelfile created!")
print("\n--- Deploy instructions ---")
print("1. Download car-parser-1.5b/ folder")
print("2. cd car-parser-1.5b/")
print("3. ollama create car-parser -f Modelfile")
print("4. ollama run car-parser")

In [ ]:
# 11. Download GGUF + Modelfile
import shutil
shutil.make_archive("car-parser-1.5b", "zip", "car-parser-1.5b")
files.download("car-parser-1.5b.zip")